In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [2]:

endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
api_key = os.getenv("AZURE_OPENAI_API_KEY")
deployment_name = "gpt-5.4-mini"

client = OpenAI(
    base_url=endpoint,
    api_key=api_key
)


In [3]:
def llm(prompt):
    response = client.responses.create(
        model=deployment_name,
        input=prompt,
    )
    return response.output

In [4]:

response = llm("can i still join the course ?")

response

[ResponseOutputMessage(id='msg_00de5ca11ff59937006a061843aad08195ac0fa2196b193512', content=[ResponseOutputText(annotations=[], text='You may be able to, but I’d need a bit more context.\n\nWhich course are you asking about, and do you mean:\n- enrolling late,\n- joining after it has started,\n- or getting on a waitlist?\n\nIf you want, I can help you draft a quick message to the course instructor or office asking if it’s still possible to join.', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]

In [5]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''


In [ ]:
question = "are you single ?"

In [7]:
prompt = f"""you are an assistant for a course, assist students by answering their questions based on the context provided. 
Say 'oops, i dont know' if the answer is not in the context. Always use the context to answer, do not use any outside information. 
Context: {context} 
Question: {question}"""

In [8]:
answer = llm(prompt)
print(answer)

[ResponseOutputMessage(id='msg_0cd615b0423206cf006a05fb52766c81949ecf9881d90cf731', content=[ResponseOutputText(annotations=[], text='oops, i dont know', type='output_text', logprobs=[])], role='assistant', status='completed', type='message', phase='final_answer')]


In [2]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [3]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course['path']}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [4]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [10]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [23]:
import pandas as pd
from pprint import pprint
df = pd.DataFrame(documents)

pprint(df['course'].unique().tolist())

['machine-learning-zoomcamp',
 'llm-zoomcamp',
 'data-engineering-zoomcamp',
 'mlops-zoomcamp']


In [26]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [28]:
question = "when will be the home works available to students?"
search_results = search(question, course='llm')
search_results

[]